[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/wu_experiment/wu_experiment.ipynb)

# The Wu Experiment

This example computes the electrons emitted by polarized cobalt-60 in the Wu experiment, by its mirror image, and by that mirror image built as a real apparatus. In 1957 Chien-Shiung Wu polarized cobalt-60 nuclei with the current in a coil and counted the electrons their decay emits. More electrons left against the nuclear spin than along it. In a mirror beside the apparatus, the current circulates the other way while the electrons still leave downward, so the mirror image shows electrons leaving along the spin. Building that mirror image for real does not reproduce it: the decay tells the two apart.

A spin is an angular momentum, and like every momentum it is modelled as intrinsic: the patch the coil's current circulates around, the dual of the coil's plane, `mv.xy.dual()`, which reverses with the current. The rate of electrons leaving along a unit direction is one plus the decay's asymmetry times the spin's pairing with that direction, `spin & direction`. Both are intrinsic, so their pairing is an intrinsic scalar, and for the rate to be an ordinary number the asymmetry is modelled as an intrinsic scalar too, a constant with a sense of its own. A mirror, `mirror >> x`, reflects every part of the apparatus, and the decay's constant is not among them.

In the notation of Wu's paper the rate reads as $1 + A\,\frac{v}{c}\cos\theta$, with $\theta$ the angle between the electron's momentum and the nuclear spin.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure
from examples.orientation.wu_experiment import render

np.set_printoptions(precision=3, suppress=True)

In [ ]:
ga = VGA3D
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Scalar = ga.gatype.scalar()                            # Scalar+
# Directions are antivectors, intrinsic like all geometry.
Direction = ga.gatype.antivector()                     # Antivector-
# A spin is an angular momentum: the patch its current circulates around, intrinsic.
Spin = ga.gatype.vector().derive.flipped_orientation   # Vector-
# A constant of the decay with a sense of its own: measured against a handedness.
Asymmetry = Scalar.derive.flipped_orientation          # Scalar-

The nuclei take their spin from the coil's current, counterclockwise seen from above. The rate along a direction departs from that of unpolarized nuclei by the decay's asymmetry times the spin's pairing with the direction, `asymmetry * (spin & directions)`. The asymmetry is minus one for cobalt-60, times the electrons' speed over the speed of light, here taken as 0.6.

The mirror stands upright beside the apparatus, the plane `mv.x`, holding the coil's axis. What the mirror shows along a direction is what the apparatus sends along that direction's mirror image. The mirrored apparatus built for real has the mirror image of the coil, `mirror >> spin`, and the same decay.

In [ ]:
# The directions around the plane holding the coil's axis, the last closing the loop on the first.
n_directions = 181
decay_asymmetry = -0.6

angles = np.linspace(0, 2 * np.pi, n_directions)
# Measured against the lab's handedness, the basis volume x ^ y ^ z.
asymmetry = mv(Asymmetry, [decay_asymmetry])                                 # [] Asymmetry-
# The patch the coil's current circulates around, counterclockwise seen from above.
spin = mv.xy.dual()                                                          # [] Spin-
# The plane x = 0, holding the coil's axis.
mirror = mv.x                                                                # [] Mirror+
# Directions turned from z toward x, around the plane that holds the coil's axis.
directions = (mv.zx * (-angles / 2)).exp() >> mv.z.dual()                    # [directions] Direction-

original = 1 + asymmetry * (spin & directions)                               # [directions] Scalar+
image = 1 + asymmetry * (spin & (mirror >> directions))                      # [directions] Scalar+
built = 1 + asymmetry * ((mirror >> spin) & directions)                      # [directions] Scalar+
# The apparatus and the mirrored apparatus, each read straight down and straight up.
coils = stack([spin, mirror >> spin])                                        # [coils] Spin-
down_and_up = stack([-mv.z.dual(), mv.z.dual()])                             # [2] Direction-
vertical_rates = 1 + asymmetry * (coils[:, None] & down_and_up)              # [coils, 2] Scalar+

In [ ]:
print("Rate down and up: the apparatus, then the mirrored apparatus")
print(vertical_rates.to_array())

Each panel below shows the rate along each direction as a distance from the centre, around the plane that holds the coil's axis, and the coil where that plane cuts it: its current goes into the page through the wire marked with a cross and comes out through the one marked with a dot. The left panel is the apparatus, the middle one its mirror image, and the right one the mirrored apparatus built for real. The middle and right panels have the same coil and send electrons opposite ways.

In [ ]:
spins = stack([spin, mirror >> spin, mirror >> spin])                        # [cases] Spin-
rates = stack([original, image, built])                                      # [cases, directions] Scalar+
save_figure(render.draw_experiment(spins, directions, rates), "wu_experiment");

Reflecting the asymmetry along with the apparatus would make the mirrored apparatus give the mirror image, as the check below confirms. A decay law built from the apparatus's own geometry with an ordinary coefficient is symmetric under every mirror; a law that is not must hold an intrinsic constant. The asymmetry is such a constant, and it belongs to the weak interaction, not to the apparatus: no mirror of the apparatus reverses it.

Neutrinos carry the same kind of constant. A neutrino's helicity, its spin's pairing with its direction of motion, `spin & direction`, is an intrinsic scalar, and every neutrino observed has helicity minus one half. The mirror image of a neutrino would have helicity plus one half, and none has been found.

In [ ]:
# checks
# With the asymmetry mirrored along, the mirrored apparatus gives the mirror image.
mirrored_law_rates = 1 + (mirror >> asymmetry) * ((mirror >> spin) & directions)   # [directions] Scalar+
np.testing.assert_allclose((mirrored_law_rates - image).to_array(), 0, atol=1e-12)
# As it is, the mirrored apparatus sends electrons where the mirror image does not.
np.testing.assert_allclose((built + image - 2).to_array(), 0, atol=1e-12)